# 1 — Ingest & Extract

Turns a source document (or pasted text) into an AI-proposed foreground process
structure and inventory, using the same two-pass extraction the *Add Foreground*
Streamlit page uses (`ai_lca.llm`): first classify every process-like entity's
role and deterministically lock only `assessed_product_system` /
`interconnected_foreground_process` candidates as foreground processes, then
extract flows constrained to that locked structure.

**This is the only notebook in this sequence that calls the OpenAI API** — it
costs tokens and can take from several seconds to a couple of minutes depending
on document length/model.

All settings come from [ai_lca_config.py](ai_lca_config.py). Edit `SOURCE_MODE`,
`SOURCE_TEXT` / `SOURCE_DOCUMENT_PATHS` and `RUN_LABEL` there, save, then run
this notebook top to bottom.

In [ ]:
from pathlib import Path

import ai_lca_config as cfg
cfg.print_config()

from ai_lca.documents import combine_document_evidence
from ai_lca.llm import extract_inventory_from_documents, extract_inventory_from_text
from ai_lca.export import candidate_structure_to_dataframe, extraction_to_dataframe, process_structure_to_dataframe
from ai_lca.notebook_helpers import run_output_dir, save_extraction, summarize_extraction

run_dir = run_output_dir(cfg.OUTPUT_DIR, cfg.RUN_LABEL)
print()
print("Run output directory:", run_dir.resolve())

## Step 1 — Load source material

Loads either the pasted text (`SOURCE_MODE = "text"`) or the documents listed
in `SOURCE_DOCUMENT_PATHS` (`SOURCE_MODE = "documents"`). Document mode also
extracts embedded figures/scanned pages as visual evidence — they get
transcribed with vision in the next step, before any LCA interpretation
happens.

In [ ]:
documents = None
source_text = None

if cfg.SOURCE_MODE == "text":
    if not cfg.SOURCE_TEXT.strip():
        raise ValueError("SOURCE_MODE='text' but ai_lca_config.SOURCE_TEXT is empty. Edit it and re-run.")
    source_text = cfg.SOURCE_TEXT
    print(f"Loaded pasted text: {len(source_text):,} character(s).")
else:
    if not cfg.SOURCE_DOCUMENT_PATHS:
        raise ValueError("SOURCE_MODE='documents' but ai_lca_config.SOURCE_DOCUMENT_PATHS is empty. Edit it and re-run.")
    documents = []
    for path in cfg.SOURCE_DOCUMENT_PATHS:
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(f"Source document not found: {p}")
        documents.append((p.name, p.read_bytes()))
    preview_text, visual_assets, ingestion_warnings = combine_document_evidence(
        documents, max_visual_assets=cfg.MAX_VISUAL_ASSETS,
    )
    print(f"Loaded {len(documents)} document(s): {', '.join(name for name, _ in documents)}")
    print(f"Combined native text: {len(preview_text):,} character(s)")
    print(f"Visual assets selected for vision transcription: {len(visual_assets)}")
    for w in ingestion_warnings:
        print("  WARNING:", w)

## Step 2 — Run extraction (calls OpenAI)

Two LLM passes happen here:

1. **Structure** — classify every process-like entity's role, then
   deterministically lock the foreground graph (see the role taxonomy in the
   project README).
2. **Flows** — extract material/energy/transport/emission/product flows,
   constrained to the just-locked process structure.

Document mode also runs a vision pass over the selected figures/scanned pages
first, folding that transcription into the source text before the two passes
above.

In [ ]:
if documents is not None:
    extraction = extract_inventory_from_documents(
        documents, model=cfg.OPENAI_MODEL, extra_instructions=cfg.EXTRA_INSTRUCTIONS,
        max_visual_assets=cfg.MAX_VISUAL_ASSETS,
    )
else:
    extraction = extract_inventory_from_text(
        source_text, model=cfg.OPENAI_MODEL, extra_instructions=cfg.EXTRA_INSTRUCTIONS,
    )
print("Extraction complete.")

## Step 3 — What got classified, locked, and extracted

In [ ]:
summarize_extraction(extraction)

### Every classified candidate — including ones *not* locked as processes, and why

`role` shows how each process-like entity in the source was classified;
`locked_as_process` shows which ones survived deterministic locking into the
foreground graph. This is the audit trail for "why wasn't X modelled as its
own process?".

In [ ]:
candidate_structure_to_dataframe(extraction)

### Locked foreground processes — note the `process_id` values, you'll need them for `PROCESS_REVIEW`

In [ ]:
process_structure_to_dataframe(extraction)

### Extracted flows — note the `flow_id` values, you'll need them for `INVENTORY_REVIEW`

In [ ]:
extraction_to_dataframe(extraction)

## Step 4 — Save for the next notebook

In [ ]:
raw_path = save_extraction(extraction, run_dir / "1_extraction_raw.json")
print("Saved raw extraction to:", raw_path)
print()
print("Next: open ai_lca_config.py, fill in PROCESS_REVIEW using the process_id values")
print("printed above (leave it empty to accept every AI-locked process as-is), save,")
print("then run 1.1.paper_process_review.ipynb.")